# imports

In [10]:
import os
import sys
import numpy as np
import pandas as pd
import mlflow
import mlflow.tensorflow

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)

# config

In [11]:
ASSET = 'BTC'
INTERVAL = '4h'
SEQUENCE_LENGTH = 20
N_FEATURES = 5
# model hyperparameters (same as old LSTM for fair comparison)
LSTM_UNITS_1 = 64
LSTM_UNITS_2 = 32
DENSE_UNITS = 16
DROPOUT_1 = 0.4
DROPOUT_2 = 0.2
LEARNING_RATE = 0.0001
EPOCHS = 50
BATCH_SIZE = 64
EARLY_STOP_PATIENCE = 15

DATA_DIR = '../../../data/processed/'

X_TRAIN_PATH = os.path.join(DATA_DIR, f'X_train_{ASSET.lower()}_{INTERVAL}_lstm_raw.npy')
X_TEST_PATH  = os.path.join(DATA_DIR, f'X_test_{ASSET.lower()}_{INTERVAL}_lstm_raw.npy')
Y_TRAIN_PATH = os.path.join(DATA_DIR, f'y_train_{ASSET.lower()}_{INTERVAL}_lstm_raw.parquet')
Y_TEST_PATH  = os.path.join(DATA_DIR, f'y_test_{ASSET.lower()}_{INTERVAL}_lstm_raw.parquet')

print(f" Asset: {ASSET} | Interval: {INTERVAL}")
print(f" Sequence length: {SEQUENCE_LENGTH} candles | Features: {N_FEATURES} log-returns")
print(f" Input files:")
print(f"   X_train: {X_TRAIN_PATH}")
print(f"   X_test:  {X_TEST_PATH}")
print(f"   y_train: {Y_TRAIN_PATH}")
print(f"   y_test:  {Y_TEST_PATH}")


 Asset: BTC | Interval: 4h
 Sequence length: 20 candles | Features: 5 log-returns
 Input files:
   X_train: ../../../data/processed/X_train_btc_4h_lstm_raw.npy
   X_test:  ../../../data/processed/X_test_btc_4h_lstm_raw.npy
   y_train: ../../../data/processed/y_train_btc_4h_lstm_raw.parquet
   y_test:  ../../../data/processed/y_test_btc_4h_lstm_raw.parquet


# mlflow setup

In [12]:
mlflow.set_tracking_uri("http://localhost:5000")

EXPERIMENT_NAME = f"{ASSET}_{INTERVAL}_deep_learning"

try:
    experiment_id = mlflow.create_experiment(EXPERIMENT_NAME)
    print(f"Created new MLflow experiment: '{EXPERIMENT_NAME}'")
except:
    experiment_id = mlflow.get_experiment_by_name(EXPERIMENT_NAME).experiment_id
    print(f"Using existing MLflow experiment: '{EXPERIMENT_NAME}'")

mlflow.set_experiment(EXPERIMENT_NAME)
print(f"   Experiment ID: {experiment_id}")

Using existing MLflow experiment: 'BTC_4h_deep_learning'
   Experiment ID: 21


# load pre-built sequences from fe notebook

In [13]:
X_train = np.load(X_TRAIN_PATH)
X_test  = np.load(X_TEST_PATH)

y_train_df = pd.read_parquet(Y_TRAIN_PATH)
y_test_df  = pd.read_parquet(Y_TEST_PATH)

y_train = y_train_df['target_direction'].values
y_test  = y_test_df['target_direction'].values

print(f"data loaded successfully")
print(f"   X_train shape: {X_train.shape}  (samples, timesteps, features)")
print(f"   X_test  shape: {X_test.shape}")
print(f"   y_train shape: {y_train.shape}")
print(f"   y_test  shape: {y_test.shape}")


data loaded successfully
   X_train shape: (10619, 20, 5)  (samples, timesteps, features)
   X_test  shape: (2655, 20, 5)
   y_train shape: (10619,)
   y_test  shape: (2655,)


# verify data integrity

In [14]:
assert X_train.shape[1] == SEQUENCE_LENGTH, f"Expected {SEQUENCE_LENGTH} timesteps, got {X_train.shape[1]}"
assert X_train.shape[2] == N_FEATURES, f"Expected {N_FEATURES} features, got {X_train.shape[2]}"
assert X_test.shape[1] == SEQUENCE_LENGTH, f"Expected {SEQUENCE_LENGTH} timesteps, got {X_test.shape[1]}"
assert X_test.shape[2] == N_FEATURES, f"Expected {N_FEATURES} features, got {X_test.shape[2]}"
assert len(y_train) == X_train.shape[0], "y_train length mismatch"
assert len(y_test) == X_test.shape[0], "y_test length mismatch"

train_pos = y_train.mean()
test_pos  = y_test.mean()

print(f"All shape assertions passed")
print(f"   Train samples: {X_train.shape[0]:,}  |  Positive class: {train_pos:.1%}")
print(f"   Test  samples: {X_test.shape[0]:,}  |  Positive class: {test_pos:.1%}")
print(f"   Sequence: {SEQUENCE_LENGTH} candles × {N_FEATURES} log-return features")
print(f"   Feature columns: open_logret, high_logret, low_logret, close_logret, volume_logret")


All shape assertions passed
   Train samples: 10,619  |  Positive class: 51.5%
   Test  samples: 2,655  |  Positive class: 50.1%
   Sequence: 20 candles × 5 log-return features
   Feature columns: open_logret, high_logret, low_logret, close_logret, volume_logret


# model architecture (2-layer LSTM — same as old for fair comparison)

In [15]:
model = Sequential([
    LSTM(LSTM_UNITS_1, return_sequences=True, input_shape=(SEQUENCE_LENGTH, N_FEATURES)),
    Dropout(DROPOUT_1),

    LSTM(LSTM_UNITS_2, return_sequences=False),
    Dropout(DROPOUT_2),

    Dense(DENSE_UNITS, activation='relu'),
    Dropout(DROPOUT_2),

    Dense(1, activation='sigmoid')
])

print(f" Model architecture (same as old LSTM for direct comparison):")
print(f"   LSTM({LSTM_UNITS_1}, return_sequences=True) → Dropout({DROPOUT_1})")
print(f"   LSTM({LSTM_UNITS_2}, return_sequences=False) → Dropout({DROPOUT_2})")
print(f"   Dense({DENSE_UNITS}, relu) → Dropout({DROPOUT_2})")
print(f"   Dense(1, sigmoid)")
model.summary()

 Model architecture (same as old LSTM for direct comparison):
   LSTM(64, return_sequences=True) → Dropout(0.4)
   LSTM(32, return_sequences=False) → Dropout(0.2)
   Dense(16, relu) → Dropout(0.2)
   Dense(1, sigmoid)


c:\Users\Manindra\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\rnn\rnn.py:204: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_4 (LSTM)                   │ (None, 20, 64)         │        17,920 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ (None, 20, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_5 (LSTM)                   │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_7 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_8 (Dropout)             │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 30,881 (120.63 KB)

 Trainable params: 30,881 (120.63 KB)

 Non-trainable params: 0 (0.00 B)